In [4]:


import os
import pandas as pd

# Path to the directory containing all .csv.gz files
base_dir = "/Users/mihokoda/Desktop/LocationReasoner/data/global-places-poi-geometry-unl_01k14d5tz91e7d0hp0vctb8a6v"

# Create lists to collect data
ny_rows = []
tampa_rows = []

# Walk through all subdirectories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".csv.gz"):
            file_path = os.path.join(root, file)
            try:
                df = pd.read_csv(file_path, compression='gzip', low_memory=False)

                if "CITY" in df.columns:
                    ny_rows.append(df[df["CITY"] == "New York"])
                    tampa_rows.append(df[df["CITY"] == "Tampa"])
            except Exception as e:
                print(f"❌ Failed to read {file_path}: {e}")


# Concatenate all matching rows
ny_df = pd.concat(ny_rows, ignore_index=True)
tampa_df = pd.concat(tampa_rows, ignore_index=True)

# Save to CSV
ny_df.to_csv("new_york_pois.csv", index=False)
tampa_df.to_csv("tampa_pois.csv", index=False)

print("✅ Files saved: new_york_pois.csv, tampa_pois.csv")

import os

# Get current working directory
output_dir = os.getcwd()

# Print full paths to the saved files
print("✅ Files saved:")
print(f"New York POIs: {os.path.join(output_dir, 'new_york_pois.csv')}")
print(f"Tampa POIs:    {os.path.join(output_dir, 'tampa_pois.csv')}")



✅ Files saved: new_york_pois.csv, tampa_pois.csv
✅ Files saved:
New York POIs: /Users/mihokoda/Desktop/LocationReasoner/code/new_york_pois.csv
Tampa POIs:    /Users/mihokoda/Desktop/LocationReasoner/code/tampa_pois.csv


In [1]:
import pandas as pd
bos = "/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Boston_POI.csv"
boscsv = pd.read_csv(bos)

In [2]:
bosfinal = "/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Boston_POI_Spend_with_zones.csv"
bosfinalcsv = pd.read_csv(bosfinal)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd

def assign_poi_kmeans(poi_df, num_zones=2000):
    coords = poi_df[['LATITUDE', 'LONGITUDE']].values

    # Run KMeans clustering
    kmeans = KMeans(n_clusters=num_zones, random_state=42)
    poi_df['zone_id'] = kmeans.fit_predict(coords)

    return poi_df


tampa = "/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/tampa_pois.csv"
tampacsv = pd.read_csv(tampa)
tampacsv = assign_poi_kmeans(tampacsv, 1500)


newyork = "/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/new_york_pois.csv"
newyorkcsv = pd.read_csv(newyork)
newyorkcsv = assign_poi_kmeans(newyorkcsv, 2300)

In [27]:
bosfinalcsv.columns

Index(['PLACEKEY', 'LOCATION_NAME', 'BRANDS', 'TOP_CATEGORY', 'SUB_CATEGORY',
       'NAICS_CODE', 'LATITUDE', 'LONGITUDE', 'STREET_ADDRESS', 'CITY',
       'REGION', 'POSTAL_CODE', 'GEOMETRY_TYPE', 'POLYGON_WKT', 'PHONE_NUMBER',
       'WKT_AREA_SQ_METERS', 'RAW_TOTAL_SPEND_2019',
       'RAW_NUM_TRANSACTIONS_2019', 'RAW_NUM_CUSTOMERS_2019',
       'MEDIAN_SPEND_PER_TRANSACTION_2019', 'MEDIAN_SPEND_PER_CUSTOMER_2019',
       'SPEND_PCT_CHANGE_VS_PREV_YEAR_2019', 'RAW_TOTAL_SPEND_2020',
       'RAW_NUM_TRANSACTIONS_2020', 'RAW_NUM_CUSTOMERS_2020',
       'MEDIAN_SPEND_PER_TRANSACTION_2020', 'MEDIAN_SPEND_PER_CUSTOMER_2020',
       'SPEND_PCT_CHANGE_VS_PREV_YEAR_2020', 'RAW_TOTAL_SPEND_2021',
       'RAW_NUM_TRANSACTIONS_2021', 'RAW_NUM_CUSTOMERS_2021',
       'MEDIAN_SPEND_PER_TRANSACTION_2021', 'MEDIAN_SPEND_PER_CUSTOMER_2021',
       'SPEND_PCT_CHANGE_VS_PREV_YEAR_2021', 'RAW_TOTAL_SPEND_2022',
       'RAW_NUM_TRANSACTIONS_2022', 'RAW_NUM_CUSTOMERS_2022',
       'MEDIAN_SPEND_PER_TRA

In [46]:
import os
import pandas as pd

def merge_spend_patterns(poi_df, base_dir):
    # Column suffixes to add from each year's file
    column_suffixes = [
        'RAW_TOTAL_SPEND',
        'RAW_NUM_TRANSACTIONS',
        'RAW_NUM_CUSTOMERS',
        'MEDIAN_SPEND_PER_TRANSACTION',
        'MEDIAN_SPEND_PER_CUSTOMER',
        'SPEND_PCT_CHANGE_VS_PREV_YEAR'
    ]
    
    # ID to merge on
    merge_key = 'PLACEKEY'
    
    # Loop over years
    for year in range(2019, 2025):
        folder = f"Spend Pattern {year}"
        path = os.path.join(base_dir, folder)
        
        # Find the .csv.gz file in the folder
        csv_gz_files = [f for f in os.listdir(path) if f.endswith('.csv.gz')]
        if not csv_gz_files:
            print(f"❌ No CSV found in {path}")
            continue
        
        file_path = os.path.join(path, csv_gz_files[0])
        print(f"📥 Loading {file_path}")
        
        # Load the full dataframe and select needed columns
        df = pd.read_csv(file_path, compression='gzip', low_memory=False)
        
        # Build expected column list
        expected_cols = [merge_key] + column_suffixes
        missing = [col for col in expected_cols if col not in df.columns]
        if missing:
            print(f"⚠️ Missing columns in {year}: {missing}")
            continue
        
        # Slice the dataframe and rename columns to include the year
        df = df[expected_cols]
        rename_map = {col: f"{col}_{year}" for col in column_suffixes}
        df = df.rename(columns=rename_map)
        
        # Merge into main POI dataframe
        poi_df = poi_df.merge(df, on=merge_key, how='left')
    
    return poi_df

base_dir = "/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset"
tampacsv = merge_spend_patterns(tampacsv, base_dir)

#newyorkcsv = merge_spend_patterns(newyorkcsv, base_dir)


📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2019/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2019-08-01.csv.gz
📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2020/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2020-08-01.csv.gz
📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2021/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2021-08-01.csv.gz
📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2022/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2022-08-01.csv.gz
📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2023/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2023-08-01.csv.gz
📥 Loading /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Spend Pattern 2024/Spend_Patterns_Entire_US-0-SPEND_DATE_RANGE_START-2024-08-01.csv.gz


In [52]:
newyorkcsv = newyorkcsv.drop(columns=[
    "PARENT_PLACEKEY",
    "CATEGORY_TAGS",
    "INCLUDES_PARKING_LOT",
    "ISO_COUNTRY_CODE",
    "POLYGON_CLASS"
], errors="ignore")


In [ ]:
newyorkcsv.to_csv("newyorkcsv.csv", index=False)
tampacsv.to_csv("tampacsv.csv", index=False)


In [62]:
import os
import pandas as pd

# Define bounding box for NYC (approximate)
TAMPA_LAT_MIN, TAMPA_LAT_MAX = 27.80, 28.20
TAMPA_LON_MIN, TAMPA_LON_MAX = -82.75, -82.20


# Input directory
path = '/Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot'

# Container for filtered rows
nyc_parking_rows = []

# Go through all CSV files
for file in os.listdir(path):
    if file.endswith(".csv") or file.endswith(".csv.gz"):
        file_path = os.path.join(path, file)
        print(f"📂 Reading: {file_path}")
        df = pd.read_csv(file_path, low_memory=False, compression='infer')
        
        # Check if latitude and longitude columns exist
        if 'LATITUDE' in df.columns and 'LATITUDE' in df.columns:
            df_filtered = df[
                (df['LATITUDE'].between(TAMPA_LAT_MIN, TAMPA_LAT_MAX)) &
                (df['LONGITUDE'].between(TAMPA_LON_MIN, TAMPA_LON_MAX))
            ]
            nyc_parking_rows.append(df_filtered)
        else:
            print('error!')

# Concatenate all NYC rows
nyc_parking_df = pd.concat(nyc_parking_rows, ignore_index=True)

# Save to CSV
nyc_parking_df.to_csv("/Users/mihokoda/Desktop/LocationReasoner/data/tampa_parking_cleaned.csv", index=False)
print("Saved to /Users/mihokoda/Desktop/LocationReasoner/data/tampa_parking_cleaned.csv")

 

📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-63.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-62.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-60.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-48.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-49.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-61.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-9.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_dataset/Parking Lot/Parking_Lots_Geometry-59.csv
📂 Reading: /Users/mihokoda/Desktop/LocationReasoner/data/safegraph_datase

Saved to /Users/mihokoda/Desktop/LocationReasoner/data/nyc_parking_cleaned.csv


In [54]:
import pandas as pd

book1 = pd.read_excel("/Users/mihokoda/Desktop/City LLM final analysis/book1.xlsx", sheet_name="react0shot")


In [55]:
index_sim = 0
index_med = 109
index_hard = 206


In [45]:
def lol(counter):
    sim = book1.iloc[index_sim:index_sim+counter]['test results']
    correct = 0
    for i in sim:
        if i == 'same':
            correct +=1
    med = book1.iloc[index_med:index_med+counter]['test results']
    for i in med:
        if i == 'same':
            correct +=1
    hard = book1.iloc[index_hard:index_hard+counter]['test results']
    for i in hard:
        if i == 'same':
            correct +=1
    return correct

In [ ]:

print(lol(40))
#0.33

57


react + reflexion
30 --> 5/30 correct --> 0.16
60 --> 15/60 correct --> 0.25
90 --> 22/90 correct --> 0.24
120 --> 37/120 correct --> 0.30
150 --> 45/150 correct --> 0.30
180 --> 60/180 correct --> 0.33
210 --> 68/210 correct --> 0.32


react
30 --> 12/30 correct --> 0.4
60 --> 23/60 correct --> 0.38
90 --> 41/90 correct --> 0.45
120 --> 43/120 correct --> 0.36
150 --> 48/150 correct --> 0.32
180 --> 54/180 correct --> 0.30
210 --> 66/210 correct --> 0.31


Gemini 2.5
30 --> 25/30 correct --> 0.83
60 --> 48/60 correct --> 0.80
90 --> 74/90 correct --> 0.82
120 --> 88/120 correct --> 0.73 
150 --> 106/150 correct --> 0.71
180 --> 125/180 correct --> 0.69
210 --> 149/210 correct --> 0.71


Gemini 1.5
30 --> 17/30 correct --> 0.57
60 --> 24/60 correct --> 0.40
90 --> 30/90 correct --> 0.34
120 --> 45/120 correct --> 0.37 
150 --> 50/150 correct --> 0.33
180 --> 62/180 correct --> 0.34
210 --> 68/210 correct --> 0.32


Gemini 2.5
